In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------
# Basic block
# ---------------------------------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, norm=True):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=not norm)]
        if norm:
            layers.append(nn.BatchNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.1, inplace=True))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


# ---------------------------------------------------
# Class 1: visual branch for raw images
# Input: raw RGB frames
# ---------------------------------------------------
class VisualBranchCNN(nn.Module):
    """
    Input:
        imgs: [B, T, 3, H, W]

    Output:
        visual_feats: [B, T, out_ch, H', W']
    """
    def __init__(self, in_ch=3, base_ch=32, out_ch=64):
        super().__init__()
        self.encoder = nn.Sequential(
            ConvBlock(in_ch, base_ch, k=3, s=2, p=1),          # H/2
            ConvBlock(base_ch, base_ch, k=3, s=1, p=1),

            ConvBlock(base_ch, base_ch * 2, k=3, s=2, p=1),    # H/4
            ConvBlock(base_ch * 2, base_ch * 2, k=3, s=1, p=1),

            ConvBlock(base_ch * 2, out_ch, k=3, s=1, p=1),
            ConvBlock(out_ch, out_ch, k=3, s=1, p=1),
        )

    def forward(self, imgs):
        B, T, C, H, W = imgs.shape
        x = imgs.reshape(B * T, C, H, W)
        x = self.encoder(x)
        _, C2, H2, W2 = x.shape
        x = x.reshape(B, T, C2, H2, W2)
        return x


# ---------------------------------------------------
# Class 2: motion / embedding branch
# Input: embeddings from encoder 1
# ---------------------------------------------------
class MotionBranchCNN(nn.Module):
    """
    Input:
        motion_feats: [B, Tm, Cm, H, W]

    Output:
        motion_out:   [B, Tm, out_ch, H, W]
    """
    def __init__(self, in_ch=128, hidden_ch=128, out_ch=64):
        super().__init__()
        self.encoder = nn.Sequential(
            ConvBlock(in_ch, hidden_ch, k=3, s=1, p=1),
            ConvBlock(hidden_ch, hidden_ch, k=3, s=1, p=1),
            ConvBlock(hidden_ch, out_ch, k=3, s=1, p=1),
            ConvBlock(out_ch, out_ch, k=3, s=1, p=1),
        )

    def forward(self, motion_feats):
        B, Tm, C, H, W = motion_feats.shape
        x = motion_feats.reshape(B * Tm, C, H, W)
        x = self.encoder(x)
        _, C2, H2, W2 = x.shape
        x = x.reshape(B, Tm, C2, H2, W2)
        return x


# ---------------------------------------------------
# Class 3: spatial-temporal fusion
# Frame-stacked 2D conv
# ---------------------------------------------------
class SpatialTemporalFusion(nn.Module):
    """
    Fuses:
      - visual features from raw frames
      - motion features from encoder 1

    Strategy:
      1. align temporal length
      2. concatenate along channel
      3. stack frames along channel dimension
      4. apply 2D convolutions for temporal fusion

    Example:
      visual_feats: [B, 4, Cv, H, W]
      motion_feats: [B, 3, Cm, H, W]

    We align visual_feats to pairwise time steps:
      use visual_feats[:, :-1]  -> [B, 3, Cv, H, W]

    Output:
      fused: [B, out_ch, H, W]
      OR sequence fused_seq: [B, Tm, hidden_ch, H, W] if needed
    """
    def __init__(self, visual_ch=64, motion_ch=64, num_pairs=3, hidden_ch=128, out_ch=128):
        super().__init__()
        self.num_pairs = num_pairs
        self.in_ch = (visual_ch + motion_ch) * num_pairs

        self.fusion = nn.Sequential(
            ConvBlock(self.in_ch, hidden_ch, k=3, s=1, p=1),
            ConvBlock(hidden_ch, hidden_ch, k=3, s=1, p=1),
            ConvBlock(hidden_ch, out_ch, k=3, s=1, p=1),
        )

    def forward(self, visual_feats, motion_feats):
        """
        visual_feats: [B, T, Cv, H, W]
        motion_feats: [B, T-1, Cm, H, W]

        returns:
            fused: [B, out_ch, H, W]
        """
        B, Tv, Cv, H, W = visual_feats.shape
        Bm, Tm, Cm, Hm, Wm = motion_feats.shape

        assert B == Bm, f"Batch size mismatch, with vision B_size {B}, motion B_size {Bm}"
        assert H == Hm and W == Wm, f"Spatial size mismatch, with vision H {H} - {Hm}, W {W} - {Wm}"

        # align raw-frame visual features to pairwise motion time steps
        # example: from 4 frames keep first 3 features to align with 3 flows
        visual_aligned = visual_feats[:, :Tm]   # [B, Tm, Cv, H, W]

        # concatenate branch outputs at each time step
        fused_seq = torch.cat([visual_aligned, motion_feats], dim=2)   # [B, Tm, Cv+Cm, H, W]

        # frame-stacked 2D conv:
        # stack temporal dimension into channel dimension
        fused_seq = fused_seq.reshape(B, Tm * (Cv + Cm), H, W)

        fused = self.fusion(fused_seq)   # [B, out_ch, H, W] !!! Tm channel is collapsed
        return fused

In [ ]:
class SpatialTemporalFusion_timeAware(nn.Module):
    """
    The same as class SpatialTemporalFusion. But returns:

    Output:
      fused: [B, Tm, out_ch, H, W]
      OR sequence fused_seq: [B, Tm, hidden_ch, H, W] if needed
    """
    def __init__(self, visual_ch=64, motion_ch=64, num_pairs=3, hidden_ch=128, out_ch=128):
        super().__init__()
        self.num_pairs = num_pairs
        self.in_ch = (visual_ch + motion_ch) * num_pairs

        self.fusion = nn.Sequential(
            ConvBlock(self.in_ch, hidden_ch, k=3, s=1, p=1),
            ConvBlock(hidden_ch, hidden_ch, k=3, s=1, p=1),
            ConvBlock(hidden_ch, out_ch, k=3, s=1, p=1),
        )

    def forward(self, visual_feats, motion_feats):
        """
        visual_feats: [B, T, Cv, H, W]
        motion_feats: [B, T-1, Cm, H, W]

        returns:
            fused: [B, out_ch, H, W]
        """
        B, Tv, Cv, H, W = visual_feats.shape
        Bm, Tm, Cm, Hm, Wm = motion_feats.shape

        assert B == Bm, f"Batch size mismatch, with vision B_size {B}, motion B_size {Bm}"
        assert H == Hm and W == Wm, f"Spatial size mismatch, with vision H {H} - {Hm}, W {W} - {Wm}"

        # align raw-frame visual features to pairwise motion time steps
        # example: from 4 frames keep first 3 features to align with 3 flows
        visual_aligned = visual_feats[:, :Tm]   # [B, Tm, Cv, H, W]

        # concatenate branch outputs at each time step
        fused_seq = torch.cat([visual_aligned, motion_feats], dim=2)   # [B, Tm, Cv+Cm, H, W]

        # frame-stacked 2D conv:
        # stack temporal dimension into channel dimension
        fused_seq = fused_seq.reshape(B * Tm, Cv + Cm, H, W)
        fused_seq = self.fusion(fused_seq)   # [B*Tm, out_ch, H, W]
        fused_seq = fused_seq.reshape(B, Tm, self.out_ch, H, W)
        
        return fused_seq